# Task 1

In [1]:
WIDTHS = [x for x in range(4, 33)] 

In [11]:
checkpoint = "prajjwal1/bert-tiny"
tokenizer_checkpoint = "bert-base-uncased"
dataset_name = "imdb"

from transformers import AutoModelForSequenceClassification

from pathlib import Path
from chop import MaseGraph

mg = MaseGraph.from_checkpoint("../saves/tutorial_2_lora")

from chop.tools import get_tokenized_dataset, get_trainer

dataset, tokenizer = get_tokenized_dataset(
    dataset=dataset_name,
    checkpoint=tokenizer_checkpoint,
    return_tokenizer=True,
)

trainer = get_trainer(
    model=mg.model,
    tokenized_dataset=dataset,
    tokenizer=tokenizer,
    evaluate_metric="accuracy",
)



# Evaluate accuracy
eval_results = trainer.evaluate()
print(f"Evaluation accuracy: {eval_results['eval_accuracy']}")

WARNING  Node finfo not found in loaded metadata.
WARNING  Node getattr_2 not found in loaded metadata.
INFO     Tokenizing dataset imdb with AutoTokenizer for bert-base-uncased.
HTTP Error 502 thrown while requesting HEAD https://huggingface.co/datasets/imdb/resolve/main/README.md
Retrying in 1s [Retry 1/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: de1c2c95-8b41-4c0d-84a9-a698d4a6f7ef)')' thrown while requesting HEAD https://huggingface.co/datasets/stanfordnlp/imdb/resolve/e6281661ce1c48d982bc483cf8a173c1bbeb5d31/.huggingface.yaml
Retrying in 1s [Retry 1/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 0530958f-c35a-48a6-aa4d-e7d6e9b7a648)')' thrown while requesting HEAD https://huggingface.co/datasets/stanfordnlp/imdb/resolve/e6281661ce1c48d982bc483cf8a173c1bbeb5d31/.huggingface.yaml
Retrying in 2s [Retry 2/5].
'(MaxRetryE

Evaluation accuracy: 0.83352


In [13]:
import chop.passes as passes

quantization_config = {
    "by": "type",
    "default": {
        "config": {
            "name": None,
        }
    },
    "linear": {
        "config": {
            "name": "integer",
            # data
            "data_in_width": 8,
            "data_in_frac_width": 4,
            # weight
            "weight_width": 8,
            "weight_frac_width": 4,
            # bias
            "bias_width": 8,
            "bias_frac_width": 4,
        }
    },
}

mg, _ = passes.quantize_transform_pass(
    mg,
    pass_args=quantization_config,
)

trainer = get_trainer(
    model=mg.model,
    tokenized_dataset=dataset,
    tokenizer=tokenizer,
    evaluate_metric="accuracy",
)
eval_results = trainer.evaluate()
print(f"Evaluation accuracy: {eval_results['eval_accuracy']}")


/home/alex/Documents/yr_4/ADLS/ELEC70109-Advanced-Deep-Learning-Systems/src/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Evaluation accuracy: 0.8158


In [14]:
# Evaluate accuracy
trainer.train()
eval_results = trainer.evaluate()
print(f"Evaluation accuracy: {eval_results['eval_accuracy']}")


Step,Training Loss
500,0.409400
1000,0.400000
1500,0.413600
2000,0.387600
2500,0.389000
3000,0.393200


Evaluation accuracy: 0.83964


In [ ]:
from pathlib import Path

mg.export("../saves/tutorial_3_qat")

## Lab 4

In [ ]:
from pathlib import Path
from chop import MaseGraph

mg = MaseGraph.from_checkpoint(f"{Path.home()}/tutorial_3_qat")

In [15]:
import chop.passes as passes

pruning_config = {
    "weight": {
        "sparsity": 0.5,
        "method": "l1-norm",
        "scope": "local",
    },
    "activation": {
        "sparsity": 0.5,
        "method": "l1-norm",
        "scope": "local",
    },
}

mg, _ = passes.prune_transform_pass(mg, pass_args=pruning_config)
trainer = get_trainer(
    model=mg.model,
    tokenized_dataset=dataset,
    tokenizer=tokenizer,
    evaluate_metric="accuracy",
    num_train_epochs=5,
)

# Evaluate accuracy
eval_results = trainer.evaluate()
print(f"Evaluation accuracy: {eval_results['eval_accuracy']}")

INFO     Pruning module: bert_encoder_layer_0_attention_self_query
INFO     Pruning module: bert_encoder_layer_0_attention_self_key
INFO     Pruning module: bert_encoder_layer_0_attention_self_value
INFO     Pruning module: bert_encoder_layer_0_attention_output_dense
INFO     Pruning module: bert_encoder_layer_0_intermediate_dense
INFO     Pruning module: bert_encoder_layer_0_output_dense
INFO     Pruning module: bert_encoder_layer_1_attention_self_query
INFO     Pruning module: bert_encoder_layer_1_attention_self_key
INFO     Pruning module: bert_encoder_layer_1_attention_self_value
INFO     Pruning module: bert_encoder_layer_1_attention_output_dense
INFO     Pruning module: bert_encoder_layer_1_intermediate_dense
INFO     Pruning module: bert_encoder_layer_1_output_dense
INFO     Pruning module: bert_pooler_dense
INFO     Pruning module: classifier
/home/alex/Documents/yr_4/ADLS/ELEC70109-Advanced-Deep-Learning-Systems/src/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is 

Evaluation accuracy: 0.74776


In [ ]:
trainer.train()
eval_results = trainer.evaluate()
print(f"Evaluation accuracy: {eval_results['eval_accuracy']}")

In [ ]:
from pathlib import Path

mg.export("../saves/tutorial_4_pruned")

## PTQ Iterations

In [7]:
import chop.passes as passes

def get_quantisation_config(width):
    quantization_config = {
        "by": "type",
        "default": {
            "config": {
                "name": None,
            }
        },
        "linear": {
            "config": {
                "name": "integer",
                # data
                "data_in_width": width,
                "data_in_frac_width": width//2,
                # weight
                "weight_width": width,
                "weight_frac_width": width//2,
                # bias
                "bias_width": width,
                "bias_frac_width": width//2,
            }
        },
    }
    return quantization_config



In [8]:


ptq_accuracies = []

for width in WIDTHS:
    mg, _ = passes.quantize_transform_pass(mg,pass_args=get_quantisation_config(width),)
    trainer = get_trainer(
        model=mg.model,
        tokenized_dataset=dataset,
        tokenizer=tokenizer,
        evaluate_metric="accuracy",
    )
    eval_results = trainer.evaluate()
    print(f"Evaluation accuracy: {eval_results['eval_accuracy']}, Bit Width: {width}")
    ptq_accuracies.append(eval_results['eval_accuracy'])

/home/alex/Documents/yr_4/ADLS/ELEC70109-Advanced-Deep-Learning-Systems/src/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Evaluation accuracy: 0.5, Bit Width: 4


/home/alex/Documents/yr_4/ADLS/ELEC70109-Advanced-Deep-Learning-Systems/src/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Evaluation accuracy: 0.5, Bit Width: 5


/home/alex/Documents/yr_4/ADLS/ELEC70109-Advanced-Deep-Learning-Systems/src/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Evaluation accuracy: 0.69216, Bit Width: 6


KeyboardInterrupt: 

In [ ]:
print(ptq_accuracies)

[0.5, 0.5, 0.69216, 0.69152, 0.80836, 0.81224, 0.82404, 0.82404, 0.83696, 0.83696, 0.83632, 0.83632, 0.83736, 0.83736, 0.83788, 0.83788, 0.83788, 0.83788, 0.83772, 0.83772, 0.83764, 0.83764, 0.83776, 0.83776, 0.83772, 0.83772, 0.83776, 0.83776, 0.83772]


## QAT Iterations

In [ ]:
qat_accuracies = []

for width in WIDTHS:
    mg, _ = passes.quantize_transform_pass(mg,pass_args=get_quantisation_config(width),)
    trainer = get_trainer(
        model=mg.model,
        tokenized_dataset=dataset,
        tokenizer=tokenizer,
        evaluate_metric="accuracy",
    )
    trainer.train()
    eval_results = trainer.evaluate()
    print(f"Evaluation accuracy: {eval_results['eval_accuracy']}, Bit Width: {width}")
    qat_accuracies.append(eval_results['eval_accuracy'])

/vol/bitbucket/rg1322/ELEC70109-Advanced-Deep-Learning-Systems/src/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
500,0.693100
1000,0.693100
1500,0.693100
2000,0.693100


KeyboardInterrupt: 

In [ ]:
print(qat_accuracies)


[0.5, 0.5, 0.83392, 0.83064, 0.84184, 0.84228, 0.84416, 0.84452, 0.84508, 0.84556, 0.84584, 0.84576, 0.84612, 0.846, 0.84544, 0.84548, 0.84596, 0.84592, 0.84596, 0.84588, 0.8458, 0.8458, 0.84584, 0.84584, 0.84608, 0.84612, 0.84596, 0.84596, 0.84604]


In [ ]:
# Find best model

qat_accuracies = [0.5, 0.5, 0.83392, 0.83064, 0.84184, 0.84228, 0.84416, 0.84452, 0.84508, 0.84556, 0.84584, 0.84576, 0.84612, 0.846, 0.84544, 0.84548, 0.84596, 0.84592, 0.84596, 0.84588, 0.8458, 0.8458, 0.84584, 0.84584, 0.84608, 0.84612, 0.84596, 0.84596, 0.84604]
ptq_accuracies = [0.5, 0.5, 0.69216, 0.69148, 0.80836, 0.81224, 0.82404, 0.82404, 0.83684, 0.83684, 0.83628, 0.83628, 0.83732, 0.83732, 0.83792, 0.83792, 0.83788, 0.83788, 0.83772, 0.83772, 0.83764, 0.83764, 0.8378, 0.8378, 0.83772, 0.83772, 0.83776, 0.83776, 0.83772]

best_PTQ = 0
best_PTQ_width = 0;
best_QAT = 0
best_QAT_width = 0;



for i in range(len(WIDTHS)):
    if ptq_accuracies[i] > best_PTQ:
        best_PTQ_width = WIDTHS[i]
    if qat_accuracies[i] > best_QAT:
        best_QAT_width = WIDTHS[i]
print(best_PTQ_width, best_QAT_width)


32 32


# Task 2


In [5]:
# PTQ 32 bit width
import numpy as np


SPARSITIES = np.arange(0.1, 1, 0.1)
pruned_accuracies = {"l1-norm": [],
                     "random" : []}
def get_pruning_config(sparsity, mode):
    pruning_config = {
        "weight": {
            "sparsity": 0.5,
            "method": mode,
            "scope": "local",
        },
        "activation": {
            "sparsity": 0.5,
            "method": mode,
            "scope": "local",
        },
    }
    return pruning_config


In [ ]:


for mode in ["l1-norm", "random"]:
    for sparsity in SPARSITIES:
        mg, _ = passes.quantize_transform_pass(mg,pass_args=get_quantisation_config(32),)
        mg.model.to("cuda:0")
        mg, _ = passes.prune_transform_pass(mg, pass_args=get_pruning_config(sparsity, mode))
        mg.model.to("cuda:0")
        trainer = get_trainer(
            model=mg.model,
            tokenized_dataset=dataset,
            tokenizer=tokenizer,
            evaluate_metric="accuracy",
            num_train_epochs=5,
        )
        trainer.train()
        # Evaluate accuracy
        eval_results = trainer.evaluate()
        print(f"Evaluation accuracy: {eval_results['eval_accuracy']}, Sparsity: {sparsity}")
        pruned_accuracies[mode].append(eval_results['eval_accuracy'])


INFO     Pruning module: bert_encoder_layer_0_attention_self_query
INFO     Pruning module: bert_encoder_layer_0_attention_self_key
INFO     Pruning module: bert_encoder_layer_0_attention_self_value
INFO     Pruning module: bert_encoder_layer_0_attention_output_dense
INFO     Pruning module: bert_encoder_layer_0_intermediate_dense
INFO     Pruning module: bert_encoder_layer_0_output_dense
INFO     Pruning module: bert_encoder_layer_1_attention_self_query
INFO     Pruning module: bert_encoder_layer_1_attention_self_key
INFO     Pruning module: bert_encoder_layer_1_attention_self_value
INFO     Pruning module: bert_encoder_layer_1_attention_output_dense
INFO     Pruning module: bert_encoder_layer_1_intermediate_dense
INFO     Pruning module: bert_encoder_layer_1_output_dense
INFO     Pruning module: bert_pooler_dense
INFO     Pruning module: classifier
/vol/bitbucket/rg1322/ELEC70109-Advanced-Deep-Learning-Systems/src/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecate

Step,Training Loss
500,0.483000
1000,0.432900
1500,0.430000
2000,0.414900
2500,0.405300
3000,0.414300
3500,0.423400
4000,0.401200
4500,0.390300
5000,0.396200


INFO     Pruning module: bert_encoder_layer_0_attention_self_query
INFO     Pruning module: bert_encoder_layer_0_attention_self_key
INFO     Pruning module: bert_encoder_layer_0_attention_self_value
INFO     Pruning module: bert_encoder_layer_0_attention_output_dense
INFO     Pruning module: bert_encoder_layer_0_intermediate_dense
INFO     Pruning module: bert_encoder_layer_0_output_dense
INFO     Pruning module: bert_encoder_layer_1_attention_self_query
INFO     Pruning module: bert_encoder_layer_1_attention_self_key
INFO     Pruning module: bert_encoder_layer_1_attention_self_value
INFO     Pruning module: bert_encoder_layer_1_attention_output_dense
INFO     Pruning module: bert_encoder_layer_1_intermediate_dense
INFO     Pruning module: bert_encoder_layer_1_output_dense
INFO     Pruning module: bert_pooler_dense
INFO     Pruning module: classifier


Evaluation accuracy: 0.83476, Sparsity: 0.1


/vol/bitbucket/rg1322/ELEC70109-Advanced-Deep-Learning-Systems/src/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


...

/vol/bitbucket/rg1322/ELEC70109-Advanced-Deep-Learning-Systems/src/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
500,0.382800
1000,0.381300
1500,0.383400
2000,0.372200
2500,0.367400
3000,0.377900
3500,0.386100
4000,0.364900
4500,0.357600
5000,0.362800


Evaluation accuracy: 0.84344, Sparsity: 0.9


INFO     Pruning module: bert_encoder_layer_0_attention_self_query
INFO     Pruning module: bert_encoder_layer_0_attention_self_key
INFO     Pruning module: bert_encoder_layer_0_attention_self_value
INFO     Pruning module: bert_encoder_layer_0_attention_output_dense
INFO     Pruning module: bert_encoder_layer_0_intermediate_dense
INFO     Pruning module: bert_encoder_layer_0_output_dense
INFO     Pruning module: bert_encoder_layer_1_attention_self_query
INFO     Pruning module: bert_encoder_layer_1_attention_self_key
INFO     Pruning module: bert_encoder_layer_1_attention_self_value
INFO     Pruning module: bert_encoder_layer_1_attention_output_dense
INFO     Pruning module: bert_encoder_layer_1_intermediate_dense
INFO     Pruning module: bert_encoder_layer_1_output_dense
INFO     Pruning module: bert_pooler_dense
INFO     Pruning module: classifier
/vol/bitbucket/rg1322/ELEC70109-Advanced-Deep-Learning-Systems/src/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecate

Step,Training Loss
500,0.694500
1000,0.693400
1500,0.691500
2000,0.690400
2500,0.690200
3000,0.689400
3500,0.688500
4000,0.684300
4500,0.678200
5000,0.668700


In [ ]:
pruned_accuracies


NameError: name 'pruned_accuracies' is not defined